# One-step diffusion posterior mean on the standard Gaussian colour-denoising benchmarks

DMID (Li et al., TPAMI 2024) holds the published state of the art on CBSD68, Kodak24 and McMaster with the frozen
ADM 256×256 unconditional model. It embeds the noisy image at the step whose noise level matches σ and averages
500 two-step samples, i.e. 1000 network evaluations per image. This notebook uses DMID's protocol (noise normalised
to std exactly σ/255, unclipped; the whole image reflect-padded to a multiple of 64) with fixed seeds, and measures:

1. one-step Tweedie at DMID's step (1 evaluation), and at the continuous step that matches σ exactly;
2. the start step calibrated on held-out DIV2K images (one factor f per σ, never tuned on a test set);
3. symmetry ensembles of the calibrated one-step estimate: ×2 (horizontal flip), ×4 (flips), ×8 (dihedral group);
4. DMID's two-step scheme without noise (η = 0), and 256-px tiles instead of the whole image (Kodak24);
5. a blind variant: the step from a wavelet noise estimate σ̂, with one multiplier calibrated on held-out images;
6. Restormer, SwinIR, SCUNet (σ-specific weights), DRUNet (given σ, or σ̂) and Restormer blind, on the same inputs;
7. blends of the ×8 ADM estimate with Restormer and SwinIR, with the weight from held-out images (and the oracle);
8. PSNR/SSIM/LPIPS of DMID's released outputs, scored with the same code;
9. the calibrated blind start step on the 48 clipped Gaussian cases of `jepa-guided-diffusion`.

Kaggle: GPU T4 ×2 (one worker thread per GPU), internet on, no attached inputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "timm", "einops"], check=True)

In [ ]:
import hashlib, importlib.util, io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, types, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from skimage.restoration import estimate_sigma

TEST_MODE = os.environ.get("TEST_MODE") == "1"     # CPU smoke test: tiny random ADM, small crops, 2 images per set
SIGMAS = [15, 25, 50]
DATASETS = ["McMaster", "Kodak24", "CBSD68"]       # smallest first; Urban100 runs separately
DMID_N = {15: 33, 25: 57, 50: 115}                 # DMID main_for_gaussian.py: start step t = N - 1
F_GRID = [0.94, 0.96, 0.98, 1.0, 1.02, 1.04, 1.06, 1.08]   # start step t(f·σ), calibrated per σ
M_GRID = [0.94, 0.97, 1.0, 1.03, 1.06, 1.09]               # blind: start step t(m·σ̂), one m for all σ
M_CLIP = [1.0, 1.02, 1.04, 1.06, 1.08, 1.1]                # clipped cases: t(m·σ̂)
TILED = {"Kodak24"}
SAVE = {"McMaster": [0, 1], "Kodak24": [0, 1, 2], "CBSD68": [0, 1, 2]}   # images whose σ = 50 outputs are saved
FUSE_W = [round(0.1 * k, 1) for k in range(11)]
N_CAL = 2 if TEST_MODE else 20                     # DIV2K valid 0801.. for calibration, two scales each
TILE, OVERLAP = (64, 16) if TEST_MODE else (256, 64)
TIME_LIMIT_H = 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())] or [torch.device("cpu")]
assert TEST_MODE or DEVICES[0].type == "cuda", "GPU required"
torch.backends.cudnn.benchmark = False             # image sizes vary
CACHE, OUT = os.environ.get("CACHE_DIR", "/tmp/cache"), "outputs"
for d in (CACHE, OUT, f"{OUT}/images"):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__, "TEST_MODE" if TEST_MODE else "")


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {path.split('/')[-1]}: {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1 · Downloads (code pinned to commits, weights from the authors' releases)

In [ ]:
t0 = time.time()
SOURCES = {
    "cszn/KAIR": ("fc1732f4a4514e42ce15e5b3a1e18c828af47a1e", ["models/basicblock.py", "models/network_unet.py"]),
    "cszn/SCUNet": ("52e440a80a655b01e0b41e9dd9bfe599bc11625e", ["models/network_scunet.py"]),
    "swz30/Restormer": ("68dc6ac472db26f16361150cb7a96a1bc87da93f", ["basicsr/models/archs/restormer_arch.py"]),
    "JingyunLiang/SwinIR": ("6545850fbf8df298df73d81f3e8cba638787c8bd", ["models/network_swinir.py"]),
}
GH = "https://github.com/{}/releases/download/{}/{}"
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": (GH.format("Li-Tong-621/DMID", "v1.0", "Datasets.zip"), 367_018_356),
    "dmid_results": (GH.format("Li-Tong-621/DMID", "v1.0", "1-Classical_gaussion.zip"), 731_281_781),
    "div2k_valid": ("https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip", 448_993_893),
    "restormer_blind": (GH.format("swz30/Restormer", "v1.0", "gaussian_color_denoising_blind.pth"), 104_611_957),
    "drunet": (GH.format("cszn/KAIR", "v1.0", "drunet_color.pth"), 130_579_305),
}
for s, scunet_size in zip(SIGMAS, (71_982_831, 71_982_761, 71_982_757)):
    FILES[f"restormer_{s}"] = (GH.format("swz30/Restormer", "v1.0", f"gaussian_color_denoising_sigma{s}.pth"), 104_611_957)
    FILES[f"swinir_{s}"] = (GH.format("JingyunLiang/SwinIR", "v0.0", f"005_colorDN_DFWB_s128w8_SwinIR-M_noise{s}.pth"), 122_905_743)
    FILES[f"scunet_{s}"] = (GH.format("cszn/KAIR", "v1.0", f"scunet_color_{s}.pth"), scunet_size)
if TEST_MODE:
    FILES.pop("adm")
for repo, (commit, paths) in SOURCES.items():
    for path in paths:
        dest = f"{CACHE}/{repo.split('/')[1]}/{path}"
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        download(f"https://raw.githubusercontent.com/{repo}/{commit}/{path}", dest)
with ThreadPoolExecutor(6) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TIMES["downloads"] = time.time() - t0
print(f"downloads {TIMES['downloads'] / 60:.1f} min")

## 2 · Test sets (DMID's release), noise, metrics

In [ ]:
def read_rgb(z, name):
    return np.asarray(Image.open(io.BytesIO(z.read(name))).convert("RGB"))


def image_key(a):
    return hashlib.sha1(a.tobytes() + str(a.shape).encode()).hexdigest()


TEST = {}
with zipfile.ZipFile(P["datasets"]) as z:
    for ds in DATASETS:
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        TEST[ds] = [(n.split("/")[-1], read_rgb(z, n)) for n in names]
if TEST_MODE:
    TEST = {ds: [(n, im[:96, :128].copy()) for n, im in v[:2]] for ds, v in TEST.items()}
print({ds: len(v) for ds, v in TEST.items()})


def noise_seed(ds, idx, sigma):
    return 1_000_000 * (DATASETS.index(ds) + 1) + 1000 * sigma + idx


def noisy_of(clean_u8, sigma, seed):
    """DMID's protocol: Gaussian noise normalised to sample std exactly sigma/255, added in float, never clipped."""
    x = torch.from_numpy(clean_u8).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


def sigma_hat_of(noisy):
    return float(estimate_sigma(noisy.permute(1, 2, 0).numpy().astype(np.float64), channel_axis=-1, average_sigmas=True))


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]   # skimage gaussian_weights=True, sigma 1.5, 11 taps


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255),
    channel-averaged: valid convolution = skimage's map after it crops (win - 1) / 2 pixels per border."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(3, 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=3)[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def to_u8(x):
    return (x[0].clamp(0, 1) * 255).round().to(torch.uint8)


def psnr_u8(a, b):
    mse = (a.double() - b.double()).pow(2).mean().item()
    return 10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf")


def scores(out_u8, ref_u8, lp):
    a, b = out_u8.double(), ref_u8.double()
    with torch.no_grad():
        lpv = lp(a.float()[None] / 127.5 - 1, b.float()[None] / 127.5 - 1).item()
    return dict(psnr=psnr_u8(out_u8, ref_u8), ssim=ssim(a, b), lpips=lpv)

## 3 · Models on every GPU
ADM as in v1 (fp16), with ADM's legacy attention computed by `scaled_dot_product_attention` (same arithmetic,
memory linear in tokens; checked against the original below). Baselines in fp32 with each paper's padding.

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet

LEGACY_ATTENTION = gd_unet.QKVAttentionLegacy.forward


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1)
    out = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2))
    return out.transpose(1, 2).reshape(bs, -1, length)


ADM_FLAGS = dict(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                 attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_scale_shift_norm=True)
if TEST_MODE:
    ADM_FLAGS.update(num_channels=32, num_head_channels=16, num_res_blocks=1)
FP16 = not TEST_MODE
ADM_STATE = None if TEST_MODE else torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_adm(device, fp16=FP16):
    torch.manual_seed(0)
    m = create_model(**ADM_FLAGS, use_fp16=fp16)
    if ADM_STATE is not None:
        m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False)
    if fp16:
        m.convert_to_fp16()
    return m


for name in [m for m in sys.modules if m == "models" or m.startswith("models.")]:
    del sys.modules[name]
sys.path.insert(0, f"{CACHE}/KAIR")
from models.network_unet import UNetRes  # needs models.basicblock from the same checkout
sys.path.remove(f"{CACHE}/KAIR")
for name in [m for m in sys.modules if m == "models" or m.startswith("models.")]:
    del sys.modules[name]  # do not shadow any other top-level "models" package
if importlib.util.find_spec("thop") is None:  # SCUNet imports thop only for a FLOP count
    sys.modules["thop"] = types.ModuleType("thop")
    sys.modules["thop"].profile = None
SCUNetNet = load_module("network_scunet", f"{CACHE}/SCUNet/models/network_scunet.py").SCUNet
RestormerNet = load_module("restormer_arch", f"{CACHE}/Restormer/basicsr/models/archs/restormer_arch.py").Restormer
SwinIRNet = load_module("network_swinir", f"{CACHE}/SwinIR/models/network_swinir.py").SwinIR


def state(path, key=None):
    sd = torch.load(path, map_location="cpu", weights_only=True)
    return sd[key] if key and key in sd else sd


def restormer():
    return RestormerNet(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4, heads=[1, 2, 4, 8],
                        ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)


def build_baselines(device):
    nets = {}
    for s in SIGMAS:
        nets[f"Restormer {s}"] = restormer()
        nets[f"Restormer {s}"].load_state_dict(state(P[f"restormer_{s}"], "params"), strict=True)
        nets[f"SwinIR {s}"] = SwinIRNet(upscale=1, in_chans=3, img_size=128, window_size=8, img_range=1.0, depths=[6] * 6,
                                        embed_dim=180, num_heads=[6] * 6, mlp_ratio=2, upsampler="", resi_connection="1conv")
        nets[f"SwinIR {s}"].load_state_dict(state(P[f"swinir_{s}"], "params"), strict=True)
        nets[f"SCUNet {s}"] = SCUNetNet(in_nc=3, config=[4] * 7, dim=64)
        nets[f"SCUNet {s}"].load_state_dict(state(P[f"scunet_{s}"]), strict=True)
    nets["Restormer blind"] = restormer()
    nets["Restormer blind"].load_state_dict(state(P["restormer_blind"], "params"), strict=True)
    nets["DRUNet"] = UNetRes(in_nc=4, out_nc=3, nc=[64, 128, 256, 512], nb=4, act_mode="R", downsample_mode="strideconv",
                             upsample_mode="convtranspose", bias=False)  # the released DPIR weights have no biases
    nets["DRUNet"].load_state_dict(state(P["drunet"]), strict=True)
    for net in nets.values():
        net.to(device).eval().requires_grad_(False)
    return nets


import lpips

CTX = [dict(dev=d, adm=make_adm(d), base=build_baselines(d), lpips=lpips.LPIPS(net="alex", verbose=False).to(d)) for d in DEVICES]
with torch.no_grad():  # the SDPA attention equals ADM's legacy attention
    probe = torch.randn(2, 3, 64, 64, device=DEVICES[0])
    tt = torch.tensor([30.0, 100.0], device=DEVICES[0])
    gd_unet.QKVAttentionLegacy.forward = LEGACY_ATTENTION
    ref = CTX[0]["adm"](probe, tt).float()
    gd_unet.QKVAttentionLegacy.forward = sdpa_attention
    ATTN_GAP = float((CTX[0]["adm"](probe, tt).float() - ref).abs().max())
print(f"max |SDPA − legacy attention| on ε: {ATTN_GAP:.2e}")
assert ATTN_GAP < 5e-2

## 4 · One-step posterior mean (Tweedie), ensembles, tiles

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)  # ADM's linear schedule
ABAR = torch.cumprod(1 - BETAS, 0)
NEG_LOG_ABAR = (-torch.log(ABAR)).numpy()


def matched(sigma01):
    """Continuous step t and abar with abar = 1 / (1 + (2σ)^2): x_t = sqrt(abar) (2y - 1) then has the model's noise level."""
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def dmid_step(sigma):
    n = DMID_N[sigma] - 1
    return float(n), float(ABAR[n])


def eps_of(model, x, t):
    """ε prediction for a batch with per-sample steps t; halves the batch on out-of-memory."""
    try:
        with torch.no_grad():
            return model(x, t)[:, :3].float()
    except torch.cuda.OutOfMemoryError:
        if x.shape[0] == 1:
            raise
        torch.cuda.empty_cache()
        h = x.shape[0] // 2
        return torch.cat([eps_of(model, x[:h], t[:h]), eps_of(model, x[h:], t[h:])])


def pad64(y):
    h, w = y.shape[-2:]
    ph, pw = (-h) % 64, (-w) % 64
    return (F.pad(y, (0, pw, 0, ph), mode="reflect") if ph or pw else y), (h, w)


def tweedie(model, y, t, ab):
    """One-step posterior mean of noisy images y [B,3,H,W] (unclipped, [0,1] units) at steps t with abar ab (per sample)."""
    t = torch.as_tensor(t, dtype=torch.float32, device=y.device).reshape(-1)
    ab = torch.as_tensor(ab, dtype=torch.float32, device=y.device).reshape(-1, 1, 1, 1)
    if y.shape[0] == 1 and t.numel() > 1:
        y = y.expand(t.numel(), -1, -1, -1)
    yp, (h, w) = pad64(y)
    x = ab.sqrt() * (2 * yp - 1)
    x0 = (x - (1 - ab).sqrt() * eps_of(model, x, t.expand(x.shape[0]))) / ab.sqrt()
    return ((x0 + 1) / 2)[..., :h, :w]


def ddim2(model, y, t, ab):
    """DMID's two steps (t -> 0 -> clean) with eta = 0: deterministic, two evaluations."""
    yp, (h, w) = pad64(y)
    x = math.sqrt(ab) * (2 * yp - 1)
    e1 = eps_of(model, x, torch.full((x.shape[0],), t, device=y.device))
    x0 = (x - math.sqrt(1 - ab) * e1) / math.sqrt(ab)
    a0 = float(ABAR[0])
    x_0 = math.sqrt(a0) * x0 + math.sqrt(1 - a0) * e1
    x0 = (x_0 - math.sqrt(1 - a0) * eps_of(model, x_0, torch.zeros(x.shape[0], device=y.device))) / math.sqrt(a0)
    return ((x0 + 1) / 2)[..., :h, :w]


D4 = [(0, False), (0, True), (2, False), (2, True), (1, False), (1, True), (3, False), (3, True)]  # (quarter turns, h-flip)


def d4_fwd(x, k, flip):
    x = torch.rot90(x, k, (-2, -1))
    return x.flip(-1) if flip else x


def d4_inv(x, k, flip):
    return torch.rot90(x.flip(-1) if flip else x, -k, (-2, -1))


def tweedie_d4(model, y, t, ab, n=8):
    """Calibrated one-step estimates of the first n dihedral transforms, mapped back; same-shape transforms share a batch."""
    outs = []
    for group in (D4[:4], D4[4:n]):
        group = group[: n - len(outs)]
        if group:
            res = tweedie(model, torch.cat([d4_fwd(y, k, f) for k, f in group]), t, ab)
            outs += [d4_inv(res[i:i + 1], k, f) for i, (k, f) in enumerate(group)]
    return outs


def tweedie_tiled(model, y, t, ab, tile=TILE, overlap=OVERLAP):
    """The same estimate from overlapping tiles blended with linear ramps."""
    _, _, H, W = y.shape
    stride = tile - overlap
    cover = lambda n: max(tile, tile + math.ceil(max(n - tile, 0) / stride) * stride)
    yp = F.pad(y, (0, cover(W) - W, 0, cover(H) - H), mode="reflect")
    r = torch.arange(tile, dtype=torch.float32, device=y.device)
    ramp = torch.clamp(torch.minimum(r + 1, tile - r) / overlap, max=1)
    w2 = ramp[:, None] * ramp[None, :]
    acc, wsum = torch.zeros_like(yp), torch.zeros_like(yp[:, :1])
    coords = [(i, j) for i in range(0, yp.shape[-2] - tile + 1, stride) for j in range(0, yp.shape[-1] - tile + 1, stride)]
    for b in range(0, len(coords), 8):
        cs = coords[b:b + 8]
        res = tweedie(model, torch.cat([yp[..., i:i + tile, j:j + tile] for i, j in cs]), t, ab)
        for (i, j), out in zip(cs, res):
            acc[..., i:i + tile, j:j + tile] += out * w2
            wsum[..., i:i + tile, j:j + tile] += w2
    return (acc / wsum)[..., :H, :W]


def run_restormer(net, y):  # test_gaussian_color_denoising.py: reflect-pad to a multiple of 8
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 8, 0, (-h) % 8), mode="reflect")
    return net(yp)[..., :h, :w]


def run_swinir(net, y):  # main_test_swinir.py: flip-pad to the next multiple of the window (always at least one row)
    h, w = y.shape[-2:]
    yp = torch.cat([y, torch.flip(y, [2])], 2)[:, :, :(h // 8 + 1) * 8, :]
    yp = torch.cat([yp, torch.flip(yp, [3])], 3)[:, :, :, :(w // 8 + 1) * 8]
    return net(yp)[..., :h, :w]


def run_drunet(net, y, sigma01):  # replicate-pad to a multiple of 8, noise-level map
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 8, 0, (-h) % 8), mode="replicate")
    return net(torch.cat([yp, torch.full_like(yp[:, :1], sigma01)], 1))[..., :h, :w]

## 5 · Calibration on held-out DIV2K (validation 0801–0820, ½ scale 512×512 crops and ¼ scale whole images)
One start-step factor f per σ (non-blind), one multiplier m on σ̂ (blind) and one blend weight per σ and partner.

In [ ]:
def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, errors)."""
    q, results, errors, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    errors.append((job, traceback.format_exc()))

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), 0
    while any(th.is_alive() for th in threads):
        time.sleep(0.5 if TEST_MODE else 20)
        if time.time() - last > 600:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(errors)} errors, {(time.time() - t0) / 60:.1f} min", flush=True)
    for job, tb in errors[:3]:
        print("ERROR", job, tb)
    return results, errors


t0 = time.time()
CAL = []
with zipfile.ZipFile(P["div2k_valid"]) as z:
    names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))[:N_CAL]
    for n in names:
        im = Image.open(io.BytesIO(z.read(n))).convert("RGB")
        half = im.resize((im.width // 2, im.height // 2), Image.LANCZOS)
        half = TF.center_crop(half, [512, 512]) if not TEST_MODE else TF.center_crop(half, [64, 96])
        quarter = im.resize((im.width // 4, im.height // 4), Image.LANCZOS)
        quarter = quarter if not TEST_MODE else TF.center_crop(quarter, [64, 64])
        CAL += [(f"{n.split('/')[-1][:-4]}_half", np.asarray(half).copy()), (f"{n.split('/')[-1][:-4]}_quarter", np.asarray(quarter).copy())]
CAL_JOBS = [(i, s) for i in range(len(CAL)) for s in SIGMAS]


def run_calibration(ctx, job):
    i, sigma = job
    name, clean_u8 = CAL[i]
    noisy = noisy_of(clean_u8, sigma, seed=7_000_000 + 1000 * sigma + i)
    y, ref = noisy[None].to(ctx["dev"]), torch.from_numpy(clean_u8).permute(2, 0, 1).to(ctx["dev"])
    s01, sh = sigma / 255, sigma_hat_of(noisy)
    rows = []
    steps = [matched(f * s01) for f in F_GRID] + [matched(m * sh) for m in M_GRID]
    outs = torch.cat([tweedie(ctx["adm"], y, [t for t, _ in steps[k:k + 4]], [a for _, a in steps[k:k + 4]]) for k in range(0, len(steps), 4)])
    for k, value in enumerate(F_GRID + M_GRID):
        kind = "f" if k < len(F_GRID) else "m"
        rows.append(dict(image=name, sigma=sigma, kind=kind, value=value, sigma_hat=sh, psnr=psnr_u8(to_u8(outs[k:k + 1]), ref)))
    with torch.no_grad():
        adm1 = outs[F_GRID.index(1.0)][None]
        for partner in ("Restormer", "SwinIR"):
            other = (run_restormer if partner == "Restormer" else run_swinir)(ctx["base"][f"{partner} {sigma}"], y)
            for w in FUSE_W:
                rows.append(dict(image=name, sigma=sigma, kind=f"blend {partner}", value=w, sigma_hat=sh,
                                 psnr=psnr_u8(to_u8(w * adm1 + (1 - w) * other), ref)))
    return rows


res, err = worker_pool(CAL_JOBS, run_calibration)
assert not err, "calibration failed"
CALIB = pd.DataFrame([r for rows in res for r in rows])
CALIB.to_csv(f"{OUT}/calibration.csv", index=False)


def best_on_grid(values, means):
    """Grid argmax refined by the parabola through it and its neighbours (clipped to them)."""
    i = int(np.argmax(means))
    if 0 < i < len(values) - 1:
        (x0, x1, x2), (y0, y1, y2) = values[i - 1:i + 2], means[i - 1:i + 2]
        den = (x0 - x1) * (x0 - x2) * (x1 - x2)
        a = (x2 * (y1 - y0) + x1 * (y0 - y2) + x0 * (y2 - y1)) / den
        b = (x2 ** 2 * (y0 - y1) + x1 ** 2 * (y2 - y0) + x0 ** 2 * (y1 - y2)) / den
        if a < 0:
            return float(np.clip(-b / (2 * a), x0, x2))
    return float(values[i])


F_HAT = {}
for s in SIGMAS:
    m = CALIB[(CALIB.kind == "f") & (CALIB.sigma == s)].groupby("value").psnr.mean()
    F_HAT[s] = best_on_grid(list(m.index), list(m.to_numpy()))
mb = CALIB[CALIB.kind == "m"].groupby(["value", "sigma"]).psnr.mean().unstack().mean(axis=1)
M_HAT = best_on_grid(list(mb.index), list(mb.to_numpy()))
W_HAT = {(p, s): float(CALIB[(CALIB.kind == f"blend {p}") & (CALIB.sigma == s)].groupby("value").psnr.mean().idxmax())
         for p in ("Restormer", "SwinIR") for s in SIGMAS}
TIMES["calibration"] = time.time() - t0
print("start-step factor f per σ:", {s: round(v, 4) for s, v in F_HAT.items()}, "| blind multiplier m:", round(M_HAT, 4))
print("blend weight of ADM:", W_HAT)
print(CALIB[CALIB.kind == "f"].pivot_table(index="value", columns="sigma", values="psnr").round(3).to_string())
print(f"calibration {TIMES['calibration'] / 60:.1f} min")

## 6 · fp16 versus fp32 ADM (DMID ran fp32): CBSD68, σ = 25, one step at DMID's t

In [ ]:
t0 = time.time()
with torch.no_grad():
    adm32 = make_adm(DEVICES[0], fp16=False)
    gaps = []
    for idx in range(min(8, len(TEST["CBSD68"]))):
        name, clean_u8 = TEST["CBSD68"][idx]
        y = noisy_of(clean_u8, 25, noise_seed("CBSD68", idx, 25))[None].to(DEVICES[0])
        ref = torch.from_numpy(clean_u8).permute(2, 0, 1).to(DEVICES[0])
        t, ab = dmid_step(25)
        gaps.append(psnr_u8(to_u8(tweedie(CTX[0]["adm"], y, t, ab)), ref) - psnr_u8(to_u8(tweedie(adm32, y, t, ab)), ref))
    del adm32
    torch.cuda.empty_cache()
FP16_GAP = dict(mean=float(np.mean(gaps)), max_abs=float(np.max(np.abs(gaps))))
print(f"PSNR(fp16) − PSNR(fp32) over {len(gaps)} images: mean {FP16_GAP['mean']:+.4f} dB, max |·| {FP16_GAP['max_abs']:.4f} dB "
      f"({time.time() - t0:.0f} s)")

## 7 · Benchmarks: every method on the same noisy images

In [ ]:
def sync(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize(dev)


def run_image(ctx, job):
    ds, idx, sigma = job
    dev, adm, base, lp = ctx["dev"], ctx["adm"], ctx["base"], ctx["lpips"]
    name, clean_u8 = TEST[ds][idx]
    noisy = noisy_of(clean_u8, sigma, noise_seed(ds, idx, sigma))
    y, ref = noisy[None].to(dev), torch.from_numpy(clean_u8).permute(2, 0, 1).to(dev)
    s01, sh = sigma / 255, sigma_hat_of(noisy)
    t_cal, ab_cal = matched(F_HAT[sigma] * s01)
    t_bl, ab_bl = matched(M_HAT * sh)
    outs, secs = {}, {}

    def timed(key, fn):
        sync(dev)
        t0 = time.time()
        with torch.no_grad():
            out = fn()
        sync(dev)
        secs[key] = time.time() - t0
        return out

    outs["ADM one step, DMID t"] = timed("ADM one step, DMID t", lambda: tweedie(adm, y, *dmid_step(sigma)))
    outs["ADM one step, t(σ)"] = timed("ADM one step, t(σ)", lambda: tweedie(adm, y, *matched(s01)))
    ens = timed("ADM-cal ×8", lambda: tweedie_d4(adm, y, t_cal, ab_cal, 8))
    for k in (1, 2, 4, 8):
        outs[f"ADM-cal ×{k}"] = torch.stack(ens[:k]).mean(0)
    outs["ADM-cal two steps (η=0)"] = timed("ADM-cal two steps (η=0)", lambda: ddim2(adm, y, t_cal, ab_cal))
    outs["ADM blind ×1"] = timed("ADM blind ×1", lambda: tweedie(adm, y, t_bl, ab_bl))
    if ds in TILED:
        outs[f"ADM-cal ×1, {TILE}-px tiles"] = timed(f"ADM-cal ×1, {TILE}-px tiles", lambda: tweedie_tiled(adm, y, t_cal, ab_cal))
    for key, fn in {"Restormer": lambda: run_restormer(base[f"Restormer {sigma}"], y),
                    "SwinIR": lambda: run_swinir(base[f"SwinIR {sigma}"], y),
                    "SCUNet": lambda: base[f"SCUNet {sigma}"](y),
                    "DRUNet": lambda: run_drunet(base["DRUNet"], y, s01),
                    "Restormer blind": lambda: run_restormer(base["Restormer blind"], y),
                    "DRUNet (σ̂)": lambda: run_drunet(base["DRUNet"], y, sh)}.items():
        outs[key] = timed(key, fn)
    rows = [dict(dataset=ds, image=name, sigma=sigma, method=m, seconds=secs.get(m), **scores(to_u8(o), ref, lp)) for m, o in outs.items()]
    rows.append(dict(dataset=ds, image=name, sigma=sigma, method="noisy input", seconds=None, **scores(to_u8(y), ref, lp)))
    fusion = []
    for partner in ("Restormer", "SwinIR"):
        for w in FUSE_W:
            fusion.append(dict(dataset=ds, image=name, sigma=sigma, partner=partner, w=w,
                               psnr=psnr_u8(to_u8(w * outs["ADM-cal ×8"] + (1 - w) * outs[partner]), ref)))
        w = W_HAT[(partner, sigma)]
        rows.append(dict(dataset=ds, image=name, sigma=sigma, method=f"blend ADM-cal ×8 + {partner} (w={w:g})", seconds=None,
                         **scores(to_u8(w * outs["ADM-cal ×8"] + (1 - w) * outs[partner]), ref, lp)))
    if sigma == 50 and idx in SAVE.get(ds, []):
        folder = f"{OUT}/images/{ds}/{idx:03d}"
        os.makedirs(folder, exist_ok=True)
        for k, (m, o) in enumerate([("clean", ref[None].float() / 255), ("noisy", y)] + list(outs.items())):
            Image.fromarray(to_u8(o).permute(1, 2, 0).cpu().numpy()).save(f"{folder}/{k:02d}-{re.sub(r'[^a-z0-9]+', '-', m.lower()).strip('-')}.png")
    return rows, fusion, dict(dataset=ds, image=name, sigma=sigma, sigma_hat=sh)


t0 = time.time()
JOBS = [(ds, i, s) for ds in DATASETS for s in SIGMAS for i in range(len(TEST[ds]))]
res, err = worker_pool(JOBS, run_image)
SCORES = pd.DataFrame([r for rows, _, _ in res for r in rows])
FUSION = pd.DataFrame([r for _, rows, _ in res for r in rows])
LEVELS = pd.DataFrame([lv for _, _, lv in res])
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
FUSION.to_csv(f"{OUT}/fusion.csv", index=False)
LEVELS.to_csv(f"{OUT}/noise_levels.csv", index=False)
TIMES["benchmarks"] = time.time() - t0
print(f"{len(res)}/{len(JOBS)} images, {len(err)} errors, {TIMES['benchmarks'] / 60:.1f} min")

## 8 · DMID's released Table-1 outputs, scored with the same code (matched to the test images by content)

In [ ]:
t0 = time.time()
KEYS = {}
with zipfile.ZipFile(P["datasets"]) as z:
    for ds in ("CBSD68", "Kodak24", "McMaster", "Urban100"):
        for n in z.namelist():
            if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/"):
                KEYS[image_key(read_rgb(z, n))] = (ds, n.split("/")[-1])
DMID_DIRS = {"CBSD68": "CBSD68", "Kodak": "Kodak24", "McMaster": "McMaster", "Urban100": "Urban100"}
with zipfile.ZipFile(P["dmid_results"]) as z:
    members = z.namelist()
    DMID_JOBS = []
    for short in DMID_DIRS:
        for n in members:
            m = re.match(rf"1-Classical_gaussion/{short}/(\d+)/denoise/(\d+)_denoise\.png$", n)
            if m:
                DMID_JOBS.append((short, int(m.group(1)), int(m.group(2)), n))
if TEST_MODE:
    DMID_JOBS = DMID_JOBS[:4]


def run_dmid(ctx, job):
    short, sigma, idx, member = job
    with zipfile.ZipFile(P["dmid_results"]) as z:
        clean = read_rgb(z, f"1-Classical_gaussion/{short}/15/clean/{idx}_clean.png")
        den = read_rgb(z, member)
    ds, name = KEYS[image_key(clean)]
    out = torch.from_numpy(den).permute(2, 0, 1).to(ctx["dev"])
    ref = torch.from_numpy(clean).permute(2, 0, 1).to(ctx["dev"])
    return dict(dataset=ds, image=name, sigma=sigma, method="DMID-d (released outputs)", seconds=None, **scores(out, ref, ctx["lpips"]))


res, err = worker_pool(DMID_JOBS, run_dmid)
DMID_SCORES = pd.DataFrame(res)
DMID_SCORES.to_csv(f"{OUT}/dmid_released_scores.csv", index=False)
TIMES["DMID outputs"] = time.time() - t0
print(f"{len(res)} DMID outputs scored, {len(err)} errors")

## 9 · Clipped Gaussian cases of `jepa-guided-diffusion`: the blind start step with a calibrated multiplier
DIV2K valid 0801–0816 (short side 256, centre crop) at σ ∈ {0.1, 0.2, 0.3}, clipped to [0, 1], v1's seeds. The
multiplier on σ̂ is chosen on images 0817–0856 (seeds as in the weight probe); the oracle uses std(noisy − clean).

In [ ]:
t0 = time.time()
n_eval, n_hold = (2, 2) if TEST_MODE else (16, 40)
with zipfile.ZipFile(P["div2k_valid"]) as z:
    names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))[:n_eval + n_hold]
    SMALL = [torch.from_numpy(np.asarray(TF.center_crop(TF.resize(Image.open(io.BytesIO(z.read(n))).convert("RGB"), 256), 256)).copy())
             .permute(2, 0, 1).float() / 255 for n in names]


def add_gaussian(x, sigma, seed=0):
    return (x + sigma * torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))).clamp(0, 1)


def clipped_case(i, j, s, held_out):
    clean = SMALL[i]
    return clean, add_gaussian(clean, s, seed=(50_000 + 100 * i + j) if held_out else 1000 * i + j)


def run_clipped(ctx, job):
    i, j, s, held_out = job
    clean, noisy = clipped_case(i, j, s, held_out)
    dev = ctx["dev"]
    y, ref = noisy[None].to(dev), to_u8(clean[None].to(dev))
    sh, sres = sigma_hat_of(noisy), float((noisy - clean).std())
    if held_out:
        steps = [matched(m * sh) for m in M_CLIP]
        outs = tweedie(ctx["adm"], y, [t for t, _ in steps], [a for _, a in steps])
        return [dict(image=i, sigma=s, multiplier=m, sigma_hat=sh, sigma_residual=sres, psnr=psnr_u8(to_u8(outs[k:k + 1]), ref))
                for k, m in enumerate(M_CLIP)]
    n = int((ABAR - 1 / (1 + 4 * s * s)).abs().argmin())
    methods = {"ADM Tweedie, v1 rule (nominal σ)": tweedie(ctx["adm"], y, float(n), float(ABAR[n])),
               "ADM Tweedie (σ̂)": tweedie(ctx["adm"], y, *matched(sh)),
               "ADM Tweedie (σ̂ × m)": tweedie(ctx["adm"], y, *matched(M_CLIP_HAT * sh)),
               "ADM Tweedie (residual σ, oracle)": tweedie(ctx["adm"], y, *matched(sres)),
               "DRUNet (σ̂)": run_drunet(ctx["base"]["DRUNet"], y, sh),
               "DRUNet (σ̂ × m)": run_drunet(ctx["base"]["DRUNet"], y, M_CLIP_HAT * sh)}
    return [dict(image=f"div2k_{i}", noise=f"gaussian {s}", method=k, sigma_hat=sh, sigma_residual=sres, **scores(to_u8(o), ref, ctx["lpips"]))
            for k, o in methods.items()]


CLIP_SIGMAS = [0.1, 0.2, 0.3]
res, err = worker_pool([(n_eval + i, j, s, True) for i in range(n_hold) for j, s in enumerate(CLIP_SIGMAS)], run_clipped)
assert not err, "clipped calibration failed"
CLIP_CAL = pd.DataFrame([r for rows in res for r in rows])
CLIP_CAL.to_csv(f"{OUT}/clipped_calibration.csv", index=False)
mc = CLIP_CAL.groupby(["multiplier", "sigma"]).psnr.mean().unstack().mean(axis=1)
M_CLIP_HAT = best_on_grid(list(mc.index), list(mc.to_numpy()))
print("clipped held-out: mean PSNR by multiplier on σ̂\n", CLIP_CAL.pivot_table(index="multiplier", columns="sigma", values="psnr").round(3).to_string(),
      f"\nchosen multiplier {M_CLIP_HAT:.4f}")
res, err = worker_pool([(i, j, s, False) for i in range(n_eval) for j, s in enumerate(CLIP_SIGMAS)], run_clipped)
CLIPPED = pd.DataFrame([r for rows in res for r in rows])
CLIPPED.to_csv(f"{OUT}/clipped_scores.csv", index=False)
TIMES["clipped cases"] = time.time() - t0
print(CLIPPED.pivot_table(index="method", columns="noise", values="psnr").round(3).to_string())

## 10 · Results

In [ ]:
PUBLISHED = {  # PSNR, from DMID Table 1 (Restormer, SwinIR, DRUNet: their papers) and MambaIR / MambaIRv2
    "DMID-d (published)": {"CBSD68": [34.45, 31.86, 28.72], "Kodak24": [35.51, 33.12, 30.14], "McMaster": [35.72, 33.49, 30.50]},
    "Restormer (published)": {"CBSD68": [34.40, 31.79, 28.60], "Kodak24": [35.47, 33.04, 30.01], "McMaster": [35.61, 33.34, 30.30]},
    "SwinIR (published)": {"CBSD68": [34.42, 31.78, 28.56], "Kodak24": [35.34, 32.89, 29.79], "McMaster": [35.61, 33.20, 30.22]},
    "DRUNet (published)": {"CBSD68": [34.30, 31.69, 28.51], "Kodak24": [35.31, 32.89, 29.86], "McMaster": [35.40, 33.14, 30.08]},
    "MambaIR (published)": {"CBSD68": [34.48, None, 28.66], "Kodak24": [35.42, 32.99, 29.92], "McMaster": [35.70, 33.43, 30.35]},
}
ALL = pd.concat([SCORES, DMID_SCORES[DMID_SCORES.dataset.isin(DATASETS)]], ignore_index=True)
for metric in ("psnr", "ssim", "lpips"):
    table = ALL.pivot_table(index="method", columns=["dataset", "sigma"], values=metric)
    for name, values in PUBLISHED.items():
        if metric == "psnr":
            for ds in DATASETS:
                for s, v in zip(SIGMAS, values[ds]):
                    table.loc[name, (ds, s)] = v
    print(f"\nmean {metric.upper()}")
    print(table.round(4 if metric != "psnr" else 3).to_string())
oracle = FUSION.groupby(["dataset", "sigma", "partner", "w"]).psnr.mean().reset_index()
print("\nblend ADM-cal ×8 + partner: mean PSNR by weight of ADM (oracle = best w per set)")
print(oracle.pivot_table(index=["partner", "w"], columns=["dataset", "sigma"], values="psnr").round(3).to_string())
speed = SCORES.dropna(subset=["seconds"]).groupby(["method", "dataset"]).seconds.mean().unstack().round(3)
print("\nseconds per image (one T4)\n", speed.to_string())
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(f_hat=F_HAT, m_hat=M_HAT, w_hat={f"{p} {s}": w for (p, s), w in W_HAT.items()}, m_clip_hat=M_CLIP_HAT,
                   fp16_gap_db=FP16_GAP, attention_gap=ATTN_GAP, minutes={k: v / 60 for k, v in TIMES.items()},
                   devices=[torch.cuda.get_device_name(d) if d.type == "cuda" else "cpu" for d in DEVICES],
                   mean=ALL.groupby(["dataset", "sigma", "method"])[["psnr", "ssim", "lpips"]].mean().reset_index().to_dict("records")),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)), f"total {(time.time() - T_START) / 60:.1f} min")